In [ ]:
import pandas as pd
import os

# ---------------------------------------------------------
# 1. DEFINE EXACT FOLDER PATHS
# Using 'r' before the string makes it a "raw string" in Python.
# This prevents Windows backslashes (\) from causing formatting errors.
# ---------------------------------------------------------
raw_path = r'C:\Olist_Ecommerce_Analytics\data\raw\\'
processed_path = r'C:\Olist_Ecommerce_Analytics\data\processed\\'

# Make sure the 'processed' folder actually exists before we try to save files there later.
# exist_ok=True means it won't crash if the folder is already there.
os.makedirs(processed_path, exist_ok=True)

# ---------------------------------------------------------
# 2. LOAD THE CSV FILES INTO PANDAS DATAFRAMES
# We are only loading the tables we actually need for our business questions.
# ---------------------------------------------------------
print("Loading datasets... this might take a few seconds.")

customers = pd.read_csv(raw_path + 'olist_customers_dataset.csv')
orders = pd.read_csv(raw_path + 'olist_orders_dataset.csv')
order_items = pd.read_csv(raw_path + 'olist_order_items_dataset.csv')
payments = pd.read_csv(raw_path + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(raw_path + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(raw_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(raw_path + 'olist_sellers_dataset.csv')
translations = pd.read_csv(raw_path + 'product_category_name_translation.csv')

# ---------------------------------------------------------
# 3. QUICK SUCCESS CHECK
# Print out the row counts just to prove everything loaded correctly.
# ---------------------------------------------------------
print("Data loaded successfully!")
print(f"Total Orders: {orders.shape[0]}")
print(f"Total Order Items: {order_items.shape[0]}")


In [ ]:
# We use a LEFT JOIN. If a product category doesn't have an English translation, 
# we keep the product but the English name will be NaN.
products = pd.merge(products, translations, on='product_category_name', how='left')

# Fill missing translations with the original Portuguese name, or 'Unknown' if both are missing
products['product_category_name_english'] = products['product_category_name_english'].fillna(products['product_category_name'])
products['product_category_name_english'] = products['product_category_name_english'].fillna('Unknown')

# Drop the original Portuguese column to keep our database clean
products = products.drop(columns=['product_category_name'])

print(products[['product_id', 'product_category_name_english']].head())

In [ ]:
# Create a dictionary of our dataframes to loop through
datasets = {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers
}

# quick sanity check: print the (rows, columns) for each dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

In [ ]:
# List of columns in the 'orders' table that need to be datetimes
date_columns = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# Check for a specific Data Quality anomaly: Delivered before purchased?
invalid_dates = orders[orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']]
print(f"Number of orders delivered before they were purchased: {len(invalid_dates)}")

# Let's remove these physically impossible rows (if any) as they ruin delivery time averages
if len(invalid_dates) > 0:
    orders = orders.drop(invalid_dates.index)

In [ ]:
# Save cleaned datasets to the processed folder
customers.to_csv(processed_path + 'cleaned_customers.csv', index=False)
orders.to_csv(processed_path + 'cleaned_orders.csv', index=False)
order_items.to_csv(processed_path + 'cleaned_items.csv', index=False)
payments.to_csv(processed_path + 'cleaned_payments.csv', index=False)
reviews.to_csv(processed_path + 'cleaned_reviews.csv', index=False)
products.to_csv(processed_path + 'cleaned_products.csv', index=False)
sellers.to_csv(processed_path + 'cleaned_sellers.csv', index=False)

print("All files cleaned and saved to the 'processed' folder!")